# CA1 Project for Machine Learning
### Oleksii Taranov
### sba26106@student.cct.ie

The dataset used in the project: **CTG.xls**

Origin of the dataset:
https://archive.ics.uci.edu/dataset/193/cardiotocography

The dataset consists of measurements of fetal heart rate (FHR) and uterine contraction (UC) features on cardiotocograms classified by expert obstetricians.

Additional Information:
2126 fetal cardiotocograms (CTGs) were automatically processed and the respective diagnostic features measured. The CTGs were also classified by three expert obstetricians and a consensus classification label assigned to each of them. Classification was both with respect to a morphologic pattern (A, B, C. ...) and to a fetal state (N, S, P). Therefore the dataset can be used either for 10-class or 3-class experiments.

In [1]:
# Install package for read Excel
!pip install xlrd

In [2]:
# import libraries
import pandas as pd
import numpy as np

In [3]:
# Load and have a look at the dataset
df = pd.read_excel('CTG.xls')

df.head()

,Unnamed: 0,Unnamed: 1,Cardiotocographic data,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,Unnamed: 10,Unnamed: 11,Unnamed: 12,Unnamed: 13
0,NaN,NaN,2126 measurements and classifications of foeta...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Worksheet,NaN,Raw Data,NaN,NaN,NaN,NaN,NaN,NaN,Data (temporal features per unit time),NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,K-W,NaN,DR is removed since p(K-W)=1
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Features,NaN,chi-square,p,NaN


By default it loaded the first sheet, that has no data. The file CTG.xls contains 3 sheets. The sheet that contains the actual data is 'Raw Data'. Load the data from the target sheet.

In [4]:
df = pd.read_excel('CTG.xls', sheet_name='Raw Data')

In [5]:
df.head()

,FileName,Date,SegFile,b,e,LBE,LB,AC,FM,UC,...,C,D,E,AD,DE,LD,FS,SUSP,CLASS,NSP
0,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Variab10.txt,1996-12-01,CTG0001.txt,240.0,357.0,120.0,120.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,9.0,2.0
2,Fmcs_1.txt,1996-05-03,CTG0002.txt,5.0,632.0,132.0,132.0,4.0,0.0,4.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,6.0,1.0
3,Fmcs_1.txt,1996-05-03,CTG0003.txt,177.0,779.0,133.0,133.0,2.0,0.0,5.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,6.0,1.0
4,Fmcs_1.txt,1996-05-03,CTG0004.txt,411.0,1192.0,134.0,134.0,2.0,0.0,6.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,6.0,1.0


The dataset loaded correctly. 
The dataset has some missing values. 
The target value is 'NSP' (1 = Normal, 2 = Suspect, 3 = Pathologic), because it is clinnically meaningfull.

In [6]:
df = df.dropna(subset=['NSP'])   # removes the rows with missing values in 'NSP' feature

In [7]:
df.head()

,FileName,Date,SegFile,b,e,LBE,LB,AC,FM,UC,...,C,D,E,AD,DE,LD,FS,SUSP,CLASS,NSP
1,Variab10.txt,1996-12-01,CTG0001.txt,240.0,357.0,120.0,120.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,9.0,2.0
2,Fmcs_1.txt,1996-05-03,CTG0002.txt,5.0,632.0,132.0,132.0,4.0,0.0,4.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,6.0,1.0
3,Fmcs_1.txt,1996-05-03,CTG0003.txt,177.0,779.0,133.0,133.0,2.0,0.0,5.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,6.0,1.0
4,Fmcs_1.txt,1996-05-03,CTG0004.txt,411.0,1192.0,134.0,134.0,2.0,0.0,6.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,6.0,1.0
5,Fmcs_1.txt,1996-05-03,CTG0005.txt,533.0,1147.0,132.0,132.0,4.0,0.0,5.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,1.0


## 2. Data Understanding

After removing the empty rows, the dataset is inspected to confirm its size, the data types of the columns and whether any missing values remain.

In [8]:
# Size of the dataset (rows, columns)
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])

# Total number of missing values left in the dataset
print('Missing values:', df.isnull().sum().sum())

Rows: 2126
Columns: 40
Missing values: 0


In [9]:
# Data types of all columns
df.dtypes.value_counts()

float64           37
str                2
datetime64[us]     1
Name: count, dtype: int64

The dataset contains 2126 records and 40 columns, which matches the description provided by the UCI Machine Learning Repository. No missing values remain after the empty rows were removed.

## 3. Feature Review

Not every column in the dataset is suitable as an input for a machine learning model. According to the *Description* sheet of the original file (Ayres de Campos et al., 2000), the columns belong to three groups:

| Group | Columns | Meaning |
|---|---|---|
| Exam data | `FileName`, `Date`, `SegFile`, `b`, `e` | Administrative information about the recording |
| Measurements | `LBE`, `LB`, `AC`, `FM`, `UC`, `ASTV`, `MSTV`, `ALTV`, `MLTV`, `DL`, `DS`, `DP`, `DR`, `Width`, `Min`, `Max`, `Nmax`, `Nzeros`, `Mode`, `Mean`, `Median`, `Variance`, `Tendency` | Values measured from the fetal heart rate signal |
| Classification | `A`, `B`, `C`, `D`, `E`, `AD`, `DE`, `LD`, `FS`, `SUSP`, `CLASS`, `NSP` | Labels assigned by expert obstetricians |

Each column is assessed with one main question: *would this value be available before the diagnosis is made?* Columns that are created by the diagnosis itself would cause **data leakage**, which means the model would learn the answer instead of learning from the signal.

### 3.1 Identifier columns

A column with a unique value in almost every row behaves as an identifier and does not carry information about the fetal state.

In [10]:
# Number of unique values in the exam data columns
exam_cols = ['FileName', 'Date', 'SegFile', 'b', 'e']
pd.DataFrame({'Unique values': df[exam_cols].nunique(),
              'Total rows': len(df)})

,Unique values,Total rows
FileName,352,2126
Date,48,2126
SegFile,2126,2126
b,979,2126
e,1064,2126


`SegFile` has a unique value for every row, so it is a pure record identifier. `FileName` and `Date` identify the patient file and the examination date, while `b` and `e` are the start and end positions of the analysed segment in the recording. None of these columns describes the condition of the fetus, therefore they are excluded.

### 3.2 Constant columns

A column with only one value cannot help to distinguish between classes.

In [11]:
# Columns with only one unique value
n_unique = df.nunique()
n_unique[n_unique == 1]

DR    1
dtype: int64

`DR` (repetitive decelerations) is equal to 0 in every record. This agrees with the note in the original file, "DR is removed since p(K-W)=1", so the column is excluded.